# Обработка и анализ табличных данных

In [ ]:
import pandas as pd

In [16]:
table_data = pd.read_excel("./data_raw/DataSet_V49 (2).xlsx")

### Обработка типов данных колонок

В ходе работы с данными выяснилось, что есть дубликаты - повторные записи пациентов в зависимости от повторных госпитализаций

Для решаемой задачи классификации это не является проблемой, так как каждая отдельная госпитализация рассматривается как отдельный случай и мы смотрим на риски неблагоприятного исхода в тот или иной момент госпитализации. 

Но, если мы начинаем разбивать нашу выборку на обучающую, тестовую и валидационную, то возникнет проблема - модель увидит одного и того же пациента в разных выборках и переобучится. Это решается с помощью стратифицированного разбиения пациентов по id (GroupKFold или train_test_split с группировкой на пациента)

In [17]:
table_data.head(3)

,Код пациента,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,BEVenMax,HCO3VenMin (b),HCO3VenMin (a),HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP
0,20-6545,Хасьянов РА,78.0,М,Да,Да,Нет,Да,NaN,NaN,...,-5.0,18.5,18.5,18.5,18.5,18.5,18.5,NaN,NaN,NaN
1,19-22109,Тюркова ГГ,80.0,Ж,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,17-13439,Ремизов РВ,40.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(table_data):
    # Sort by column: 'Sex' (ascending)
    table_data = table_data.sort_values(['Sex'], na_position='first')
    return table_data

table_data_clean = clean_data(table_data.copy())
table_data_clean.head()

,Код пациента,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,BEVenMax,HCO3VenMin (b),HCO3VenMin (a),HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP
47,17-12105,Бабаев СН,59.0,NaN,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
106,17-17180,Ползиков СВ,58.0,NaN,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
158,15536,Анфёров ЮА,74.0,NaN,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
461,21-8249,Арутюнян МА,66.0,NaN,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
568,17-20825,Кондрашин АА,31.0,NaN,Нет,Да,Нет,Нет,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
counts = table_data["Name"].value_counts()
duplicated_names = counts[counts > 1].index.to_list()
duplicated_df = table_data[table_data["Name"].isin(duplicated_names)]

duplicated_df

,Код пациента,Name,Age,Sex,Наличие в БД,Наличие в файле,STEMI,ЧКВ,Дата STEMI,Вид STEMI,...,BEVenMax,HCO3VenMin (b),HCO3VenMin (a),HCO3VenMin,HCO3VenMax (b),HCO3VenMax (a),HCO3VenMax,BNP (b),BNP (a),BNP
0,20-6545,Хасьянов РА,78.0,М,Да,Да,Нет,Да,NaN,NaN,...,-5.0,18.5,18.5,18.5,18.5,18.5,18.5,NaN,NaN,NaN
11,975,Щетинкин ЮВ,56.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,590,Вирич ВИ,75.0,М,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30,1722,Бабич АП,65.0,Ж,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,17-12105,Бабаев СН,59.0,NaN,Да,Да,Нет,Да,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17419,10407,Ефименко ВИ,64.0,М,Да,Да,Да,Да,2015-07-16 00:00:00,Задний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17425,10019,Гаевский АИ,60.0,М,Да,Да,Да,Да,2021-06-08 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17427,1,Халиман ОА,48.0,М,Да,Да,Да,Да,2020-01-01 00:00:00,Передний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17428,2024-06-18 00:00:00,Журавлев АЮ,50.0,М,Да,Да,Да,Да,2018-01-01 00:00:00,Задний,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Всего записей-дубликатов - около 5к. 

Почти все из них - это повторные госпитализации. Вероятно, среди них есть и полные дубликаты записей, и некорректные строки. 
Все такие записи имеют уникальный код пациента

## Обработка данных

В ходе анализа датасета были выявлены МНОГОЧИСЛЕННЫЕ ошибки в формировании и заполнении датасета. По возможности, эти проблемы требуется исправить вручную

Обработка пропусков в колонке Пол. Данные заполнялись по сколнениям фамилии только там, где была уверенность в принадлежности к тому или иному полу. "Пограничные" случаи оставались None

In [ ]:
# Loaded variable 'table_data' from kernel state

# Sort by column: 'Sex' (ascending)
table_data = table_data.sort_values(['Sex'], na_position='first')

table_data.loc[47, "Sex"] = "М"
table_data.loc[106, "Sex"] = "М"
table_data.loc[158, "Sex"] = "М"
table_data.loc[568, "Sex"] = "М"
table_data.loc[882, "Sex"] = "М"
table_data.loc[1064, "Sex"] = "М"
table_data.loc[1197, "Sex"] = "М"
table_data.loc[1241, "Sex"] = "М"
table_data.loc[1513, "Sex"] = "М"
table_data.loc[1871, "Sex"] = "М"
table_data.loc[2420, "Sex"] = "М"
table_data.loc[2423, "Sex"] = "М"
table_data.loc[3133, "Sex"] = "Ж"
table_data.loc[3406, "Sex"] = "М"
table_data.loc[4306, "Sex"] = "Ж"
table_data.loc[5320, "Sex"] = "М"
table_data.loc[6157, "Sex"] = "Ж"
table_data.loc[6639, "Sex"] = "Ж"
table_data.loc[6998, "Sex"] = "М"
table_data.loc[7138, "Sex"] = "М"
table_data.loc[7240, "Sex"] = "М"
table_data.loc[7468, "Sex"] = "М"
table_data.loc[7548, "Sex"] = "М"
table_data.loc[9072, "Sex"] = "М"
table_data.loc[9073, "Sex"] = "Ж"
table_data.loc[9325, "Sex"] = "М"
table_data.loc[11323, "Sex"] = "М"
table_data.loc[11330, "Sex"] = "Ж"
table_data.loc[11526, "Sex"] = "М"
table_data.loc[11703, "Sex"] = "М"
table_data.loc[11822, "Sex"] = "Ж"
table_data.loc[11831, "Sex"] = "М"
table_data.loc[11919, "Sex"] = "М"
table_data.loc[11934, "Sex"] = "Ж"
table_data.loc[12078, "Sex"] = "М"
table_data.loc[12182, "Sex"] = "М"
table_data.loc[11216, "Sex"] = "М"
table_data.loc[11272, "Sex"] = "М"
table_data.loc[12471, "Sex"] = "М"
table_data.loc[12552, "Sex"] = "Ж"
table_data.loc[15950, "Sex"] = "М"

In [20]:
table_data["Стентирование в анамнезе"].value_counts()

Стентирование в анамнезе
nan    12666
0       3400
1       1364
Name: count, dtype: int64

In [19]:
table_data['Стентирование в анамнезе'] = table_data['Стентирование в анамнезе'].astype(str).str.strip().str.lower()

mapping = {
    'да': 1,
    'нет': 0,
    '1': 1,
    '0': 0
}

# Только то, что перечислено, будет заменено. Остальное остаётся как есть.
table_data['Стентирование в анамнезе'] = table_data['Стентирование в анамнезе'].replace(mapping)

In [21]:
table_data["Форма ФП"].value_counts()

Форма ФП
Пароксизм ФП                                                        1002
Постоянная форма ФП                                                  713
Пароксизмальная форма ФП Пароксизм ФП                                379
Пароксизмальная форма ФП                                             364
Персистирующая форма ФП                                               43
                                                                    ... 
Пароксизмальная форма ФП Пароксизм ФП Синдром Фредерика                1
Персистирующая форма ФП, впервые пароксизм, ритм не восстановлен       1
Пароксизм ФП, брадиформа, синдром Фредерика                            1
Персистирующая форма ФП Пароксизм ФП Синдром Фредерика                 1
Пакросизм ФП                                                           1
Name: count, Length: 63, dtype: int64

In [22]:
mapping = {
    'ПароксизмФП': 'Пароксизм ФП',
    'Пароксизм фп': 'Пароксизм ФП',
    'Пароксизмальная форма фп': 'Пароксизмальная форма ФП',
}

table_data['Форма ФП'] = table_data['Форма ФП'].replace(mapping)

Форма ФП
Пароксизм ФП                                                                         1007
Постоянная форма ФП                                                                   713
Пароксизмальная форма ФП Пароксизм ФП                                                 379
Пароксизмальная форма ФП                                                              366
Персистирующая форма ФП                                                                43
Пароксизмальная форма ФП Постоянная форма ФП                                           32
Постоянная форма ФП Персистирующая форма ФП                                            31
Постоянная форма ФП Синдром Фредерика                                                  31
Постоянная форма ФП Пароксизм ФП                                                       27
Персистирующая форма ФП Пароксизм ФП                                                   21
Пароксизмальная форма ФП Постоянная форма ФП Пароксизм ФП                              17
П